# PRATYAKSHA · Step 3 — R3: give the AI the right information, computed

**What Step 2 showed:** small models cannot add or subtract, so they listed every invoice instead of choosing.
**What R3 changes:** code computes which invoices explain the amount (exactly, minus TDS, or slightly short),
adds the payment e-mails and any identical earlier credit, and the AI just **picks one option**.

**The fair test:** a no-AI *control* gets the same options and always picks the first one.
The AI earns its place only if it beats that control.

Run top to bottom. Runtime: **T4 GPU**. Upload `pratyaksha_step3.zip` to *Colab Notebooks* first.

## 1. Connect Drive
**Why:** code and results live in Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Unpack Step 3 code
**Why:** adds R3 without touching earlier results.

In [2]:
%cd "/content/drive/MyDrive/Colab Notebooks"
!unzip -o -q pratyaksha_step3.zip
%cd "/content/drive/MyDrive/Colab Notebooks/pratyaksha"
!ls pratyaksha/systems

/content/drive/MyDrive/Colab Notebooks
/content/drive/MyDrive/Colab Notebooks/pratyaksha
__init__.py  llm_ladder.py  __pycache__  r3_context.py	rules_baseline.py


In [3]:
import json, glob
run = sorted(glob.glob("runs/*R3_Qwen3.5-4B_dev_clean/predictions.jsonl"))[-1]
rows = [json.loads(l) for l in open(run)]
for r in rows[:3]:
    print("-----"); print(r["pred"]["raw_output"][:300])

-----
Thinking Process:

1.  **Analyze the Request:**
    *   Task: Reconcile a bank-statement line for Wide World Importers India Pvt Ltd.
    *   Input Data:
        *   Bank Line: Date 2026-07-30, Credit Rs 27,217.05, Narration "CH
-----
Thinking Process:

1.  **Analyze the Request:**
    *   Task: Reconcile a bank-statement line for Wide World Importers India Pvt Ltd.
    *   Input Data:
        *   Bank Line: Date 2026-07-30, Credit Rs 52,886.42, Narration: U
-----
Thinking Process:

1.  **Analyze the Request:**
    *   Task: Reconcile a bank-statement line for Wide World Importers India Pvt Ltd.
    *   Input Data:
        *   Bank Line: Date 2026-07-30, Credit Rs 1,41,033.60, Narration


## 3. Install
**Why:** a new Colab session starts empty.

In [3]:
!pip install -q -U transformers accelerate sentence-transformers rank-bm25 pytest

## 4. Data and tests
**Expect:** `26 passed`. **Why:** checks that the computed options never use future invoices and always add up.

In [4]:
!python -m pratyaksha.world.generate --out data/wwi_in
!python -m pytest -q

{"customers": 60, "suppliers": 8, "invoices": 1361, "bank_entries": 997, "remittances": 501}
..........................                                               [100%]
26 passed in 2.38s


## 5. The no-AI control (R3 context, always pick the first option)
**Why:** this is the bar the AI must beat. It uses the real reranker (GPU), but no language model. ~2 min.
**Watch:** *Right answer among computed options* = the best any model could do with these options.

In [5]:
for cond in ["clean", "stress"]:
    !python -m pratyaksha.harness.runner --system r3_control --split dev --condition {cond} --cost-mode gpu

config.json: 100% 795/795 [00:00<00:00, 3.73MB/s]

model.safetensors: downloading bytes:  13% 292M/2.27G [00:02<00:17, 114MB/s, 25.0MB/s  ]
model.safetensors: downloading bytes:  15% 350M/2.27G [00:02<00:14, 132MB/s, 28.5MB/s  ]
model.safetensors: downloading bytes:  18% 411M/2.27G [00:03<00:12, 145MB/s, 33.2MB/s  ]
model.safetensors: downloading bytes:  41% 924M/2.27G [00:06<00:06, 202MB/s, 64.9MB/s  ]
model.safetensors: downloading bytes:  51% 1.16G/2.27G [00:08<00:06, 179MB/s, 72.8MB/s  ]
model.safetensors: downloading bytes:  66% 1.50G/2.27G [00:14<00:03, 229MB/s, 70.1MB/s  ]
model.safetensors: reconstructing file:  77% 1.74G/2.27G [00:15<00:04, 114MB/s, 87.4MB/s  ]
model.safetensors: downloading bytes: 100% 1.52G/1.52G [00:19<00:00, 76.4MB/s, 73.2MB/s  ]
model.safetensors: reconstructing file: 100% 2.27G/2.27G [00:19<00:00, 114MB/s,  110MB/s  ]
Loading weights: 100% 393/393 [00:00<00:00, 13255.72it/s]
tokenizer_config.json: 100% 1.17k/1.17k [00:00<00:00, 3.15MB/s]

sentencepiece.b

## 6. R1 with the 2B model (fair R1 vs R2 comparison)
**Why:** in Step 2 the 0.8B model scored ~0% in both R1 and R2, so we could not tell whether reranking helps.
2B works, so we run R1 at 2B and compare with the R2-2B result from Step 2. ~6 min each.

In [6]:
for cond in ["clean", "stress"]:
    !python -m pratyaksha.harness.runner --system r1 --split dev --condition {cond} --cost-mode gpu \
        --opt model=Qwen/Qwen3.5-2B --opt batch=8 --opt k=5

preprocessor_config.json: 100% 390/390 [00:00<00:00, 1.89MB/s]
chat_template.jinja: 100% 7.75k/7.75k [00:00<00:00, 14.8MB/s]
config.json: 100% 2.91k/2.91k [00:00<00:00, 7.18MB/s]
tokenizer_config.json: 100% 16.7k/16.7k [00:00<00:00, 34.5MB/s]

tokenizer.json: downloading bytes:  11% 1.35M/12.8M [00:00<00:05, 2.11MB/s]
tokenizer.json: downloading bytes: 100% 5.68M/5.68M [00:00<00:00, 7.24MB/s,  556kB/s  ]
tokenizer.json: reconstructing file: 100% 12.8M/12.8M [00:00<00:00, 16.3MB/s, 1.25MB/s  ]
video_preprocessor_config.json: 100% 385/385 [00:00<00:00, 1.82MB/s]
model.safetensors.index.json: 100% 64.5k/64.5k [00:00<00:00, 138MB/s]

model.safetensors-00001-of-00001.safeten(…): downloading bytes:   4% 190M/4.55G [00:01<00:20, 208MB/s, 17.6MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:  20% 928M/4.55G [00:04<00:21, 171MB/s, 72.3MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:  30% 1.38G/4.55G [00:06<00:11, 288MB/s, 96.7MB/s  ]
model.safetensor

## 7. R3 with the 0.8B model
**Why:** with the maths done for it, can even the smallest model now work?

In [7]:
for cond in ["clean", "stress"]:
    !python -m pratyaksha.harness.runner --system r3 --split dev --condition {cond} --cost-mode gpu \
        --opt model=Qwen/Qwen3.5-0.8B --opt batch=8

preprocessor_config.json: 100% 390/390 [00:00<00:00, 1.46MB/s]
chat_template.jinja: 100% 7.75k/7.75k [00:00<00:00, 24.5MB/s]
config.json: 100% 2.91k/2.91k [00:00<00:00, 5.11MB/s]
tokenizer_config.json: 100% 16.7k/16.7k [00:00<00:00, 53.3MB/s]
video_preprocessor_config.json: 100% 385/385 [00:00<00:00, 2.24MB/s]
model.safetensors.index.json: 100% 50.9k/50.9k [00:00<00:00, 101MB/s]

model.safetensors-00001-of-00001.safeten(…): downloading bytes:  10% 177M/1.75G [00:01<00:08, 183MB/s, 14.6MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:  17% 289M/1.75G [00:02<00:09, 157MB/s, 25.1MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:  37% 653M/1.75G [00:04<00:06, 175MB/s, 51.9MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:  60% 1.05G/1.75G [00:07<00:07, 99.2MB/s, 70.9MB/s  ]
model.safetensors-00001-of-00001.safeten(…): downloading bytes:  85% 1.49G/1.75G [00:09<00:00, 298MB/s, 91.7MB/s  ]
model.safetensors-00001-of-00001.safete

## 8. R3 with the 2B model
**Why:** same question at the size that worked in Step 2.

In [8]:
for cond in ["clean", "stress"]:
    !python -m pratyaksha.harness.runner --system r3 --split dev --condition {cond} --cost-mode gpu \
        --opt model=Qwen/Qwen3.5-2B --opt batch=8

Loading weights: 100% 617/617 [00:18<00:00, 34.10it/s]
Loading weights: 100% 393/393 [00:00<00:00, 3714.92it/s]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-att

## 9. (Optional) R3 with the 4B model
**Why:** does a bigger model read the e-mails better (bank-detail changes, which invoices)? ~8 GB download.
Batch 4 to fit memory.

In [9]:
for cond in ["clean", "stress"]:
    !python -m pratyaksha.harness.runner --system r3 --split dev --condition {cond} --cost-mode gpu \
        --opt model=Qwen/Qwen3.5-4B --opt batch=4

preprocessor_config.json: 100% 390/390 [00:00<00:00, 1.48MB/s]
chat_template.jinja: 100% 7.76k/7.76k [00:00<00:00, 20.0MB/s]
config.json: 100% 3.16k/3.16k [00:00<00:00, 12.9MB/s]
tokenizer_config.json: 100% 16.7k/16.7k [00:00<00:00, 51.0MB/s]
video_preprocessor_config.json: 100% 385/385 [00:00<00:00, 1.27MB/s]
model.safetensors.index.json: 100% 76.2k/76.2k [00:00<00:00, 156MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/5.33G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/9.32G [00:00<?, ?B/s]
Reconstructing (incomplete total...):  26% 2.41G/9.32G [00:23<01:33, 73.8MB/s,  119MB/s  ]
Reconstructing (incomplete total...):  91% 8.51G/9.32G [01:31<00:12, 65.2MB/s,  184MB/s  ]
Reconstructing (incomplete total...):  96% 8.95G/9.32G [01:42<00:07, 46.9MB/s, 58.6MB/s  ]
Reconstructing (incomplete total...): 100% 9.32G/9.32G [01:43<00:00, 80.

## 10. Compare all runs
**Why:** this table is the Step 3 result.

In [10]:
!python scripts/compare.py --split dev

system                      cond        n     acc    auto  autoErr   p50 s  party@k  tok_in parseErr   bar
R3_control_first_option     clean     177   89.8%   82.5%     0.0%    0.10    95.9%     339        0  FAIL
R3_control_first_option     stress    177   88.1%   80.8%     0.0%    0.11    95.3%     341        0  FAIL
R1_Qwen3.5-2B               clean     177   45.8%   81.9%    45.5%    1.92    90.6%    1929       22  FAIL
R1_Qwen3.5-2B               stress    177   40.7%   80.8%    50.3%    1.95    88.3%    1907       21  FAIL
R3_Qwen3.5-0.8B             clean     177   63.8%   57.1%     0.0%    0.75    95.9%     526        4  FAIL
R3_Qwen3.5-0.8B             stress    177   61.6%   55.4%     0.0%    0.78    95.3%     527        6  FAIL
R3_Qwen3.5-2B               clean     177   89.8%   26.0%     0.0%    0.73    95.9%     526        1  FAIL
R3_Qwen3.5-2B               stress    177   88.7%   19.8%     0.0%    0.76    95.3%     527        2  FAIL
R3_Qwen3.5-4B               clean    

## 11. What did the AI add over the control?
**Why:** shows the exact lines where the AI fixed or broke the control's answer. This is the evidence
for "what the model buys". (Edit `R3_Qwen3.5-2B` to `R3_Qwen3.5-4B` if you ran cell 9.)

In [11]:
for cond in ["clean", "stress"]:
    !python scripts/vs_control.py --model R3_Qwen3.5-2B --condition {cond}

R3_Qwen3.5-2B vs R3_control_first_option (clean, n=177): model fixes 5, model breaks 5

== MODEL RIGHT, CONTROL WRONG (first 5)
- B00651 clubbed_payment  'NEFT-UTIBN917338193380-NAVKAR GENERAL STORES'
    model said: '{"option": "M2", "needs_review": true, "confidence": 0.85, "reason": "Clubbed payment for multiple invoices."}\n'
- B00662 tds_deduction    'IMPS/P2A/581678784449/SHREEGANESHT/FEDERAL'
    model said: '{\n  "option": "M2",\n  "needs_review": true,\n  "confidence": 0.85,\n  "reason": "TDS deduction on Shree Ganesh Trading Co"'
- B00698 clubbed_payment  'NEFT-KKBKN325062764994-SRINIVASA STORES-WWI/26-27/00821'
    model said: '{"option": "M3", "needs_review": true, "confidence": 0.95, "reason": "Clubbed payment for multiple invoices."}\n'
- B00711 duplicate_credit 'IMPS/P2A/735702197674/SHREEGANESHT/FEDERAL/WWI/26-27/00'
    model said: '{"option": "D", "needs_review": true, "confidence": 0.95, "reason": "Duplicate credit on same date for same invoice."}\n'
- B00778 duplica

## 12. Send back
Download the notebook (*File → Download → .ipynb*) and send it, as you did for Step 2.